# Delay Locked Loop Explainer

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SiliconJackets/Cac_Spring26/blob/main/DelayLockedLoop.ipynb)

```
Copyright 2026 SiliconJackets @ Georgia Institute of Technology
SPDX-License-Identifier: Apache-2.0
```

|Name|Affiliation| Email |IEEE Member|SSCS Member|
|:--:|:----------:|:----------:|:----------:|:----------:|
|Ethan Huang|Georgia Institute of Technology|ethanhuang@gatech.edu|No|No|
|Zheng-Yin Lee|Georgia Institute of Technology|zlee63@gatech.edu|No|No|
|Alfi Misha Antony Selvin Raj|Georgia Institute of Technology|alfiselvin@gatech.edu|No|No|
|Oliver S. Lee|Georgia Institute of Technology|xli3086@gatech.edu|Yes|No|
|Shreyas Angadi|Georgia Institute of Technology|sangadi6@gatech.edu|No|No|
|Mythri Muralikannan|Georgia Institute of Technology|mmuralikannan3@gatech.edu|Yes|Yes|

This notebook provides an interactive exploration of a Delay Locked Loop (DLL), illustrating the functionality and design variations of key submodules: the Phase Detector, Controller, and the DCDL. It analyzes the behavior and trade-off associated with each component with SPICE simulations. The notebook culminates in multiple example use cases of DLLs with the appropriate submodules are chosen through our preceding analysis. The design targets the [open source sky130 PDK](https://github.com/google/skywater-pdk/) by Google and Skywater.

In [1]:
# @title Download Dependencies {display-mode: "form"}
# @markdown
# @markdown Click the ▷ button to download all neccesary dependencies (librelane + nix-os)
# @markdown as well as all of our RTL files + scripts for this interactive notebook
# @markdown
import os
from pathlib import Path
import subprocess
import sys
import shutil
import tempfile

# NgSpice Installation
!sudo apt-get update
!sudo apt-get install -y ngspice

# Nix Installation
os.environ["LOCALE_ARCHIVE"] = "/usr/lib/locale/locale-archive"

if "google.colab" in sys.modules:
    if shutil.which("nix-env") is None:
        with tempfile.TemporaryDirectory() as d:
            d = Path(d)
            installer_path = d / "nix"
            !curl --proto '=https' --tlsv1.2 -sSf -L https://install.determinate.systems/nix > {installer_path}
            with subprocess.Popen(
                [
                    "bash",
                    installer_path,
                    "install",
                    "--prefer-upstream-nix",
                    "--no-confirm",
                    "--extra-conf",
                    "extra-substituters = https://nix-cache.fossi-foundation.org\nextra-trusted-public-keys = nix-cache.fossi-foundation.org:3+K59iFwXqKsL7BNu6Guy0v+uTlwsxYQxjspXzqLYQs=\n",
                ],
                stdout=subprocess.PIPE,
                stderr=subprocess.STDOUT,
                encoding="utf8",
            ) as p:
                for line in p.stdout:
                    print(line, end="")
else:
    if shutil.which("nix-env") is None:
        raise RuntimeError("Nix is not installed!")

os.environ["PATH"] = f"/nix/var/nix/profiles/default/bin/:{os.getenv('PATH')}"


# Librelane Installation
import os
import yaml
import subprocess
import IPython

librelane_version = "latest"  # @param {key:"LibreLane Version", type:"string"}

if librelane_version == "latest":
    librelane_version = "main"

pdk_root = "~/.ciel"  # @param {key:"PDK Root", type:"string"}

pdk_root = os.path.expanduser(pdk_root)

pdk = "sky130"  # @param {key:"PDK (without the variant)", type:"string"}

librelane_ipynb_path = os.path.join(os.getcwd(), "librelane_ipynb")

display(IPython.display.HTML("<h3>Downloading LibreLane…</a>"))


TESTING_LOCALLY = False
!rm -rf {librelane_ipynb_path}
!mkdir -p {librelane_ipynb_path}
if TESTING_LOCALLY:
    !ln -s {os.getcwd()} {librelane_ipynb_path}
else:
    !curl -L "https://github.com/librelane/librelane/tarball/{librelane_version}" | tar -xzC {librelane_ipynb_path} --strip-components 1

try:
    import tkinter
except ImportError:
    if "google.colab" in sys.modules:
        !sudo apt-get install python-tk

try:
    import tkinter
except ImportError as e:
    display(
        IPython.display.HTML(
            '<h3 style="color: #800020";>❌ Failed to import the <code>tkinter</code> library for Python, which is required to load PDK configuration values. Make sure <code>python3-tk</code> or equivalent is installed on your system.</a>'
        )
    )
    raise e from None


display(IPython.display.HTML("<h3>Downloading LibreLane's dependencies…</a>"))
try:
    with subprocess.Popen(
        [
            "nix",
            "profile",
            "install",
            ".#colab-env",
        ],
        cwd=librelane_ipynb_path,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        encoding="utf8",
    ) as p:
        for line in p.stdout:
            print(line, end="")
except subprocess.CalledProcessError as e:
    display(
        IPython.display.HTML(
            '<h3 style="color: #800020";>❌ Failed to install binary dependencies using Nix…</h3>'
        )
    )

display(IPython.display.HTML("<h3>Downloading Python dependencies using PIP…</a>"))
try:
    subprocess.check_call(
        ["pip3", "install", "."],
        cwd=librelane_ipynb_path,
    )
except subprocess.CalledProcessError as e:
    display(
        IPython.display.HTML(
            '<h3 style="color: #800020";>❌ Failed to install Python dependencies using PIP…</h3>'
        )
    )
    raise e from None

display(IPython.display.HTML("<h3>Downloading PDK…</a>"))
import ciel
from ciel.source import StaticWebDataSource

with open(
    os.path.join(librelane_ipynb_path, "librelane", "pdk_hashes.yaml"), "r"
) as file:
    pdk_hashes = yaml.safe_load(file)

ciel.enable(
    ciel.get_ciel_home(pdk_root),
    pdk,
    pdk_hashes[pdk],
    data_source=StaticWebDataSource("https://fossi-foundation.github.io/ciel-releases"),
)

sys.path.insert(0, librelane_ipynb_path)
display(IPython.display.HTML("<h3>⭕️ Done.</a>"))

import logging

# Remove the stupid default colab logging handler
logging.getLogger().handlers.clear()

#Install our scripts
%cd /content/
!rm -rf CAC_2026
!git clone https://github.com/SiliconJackets/CaC_Spring26 CAC_2026

#Install iverilog
!sudo apt-get install -y iverilog

#Install VCD reader
!pip install vcdvcd -q

# Setup all neccesary graphic design parts
import sys
import os
from pathlib import Path

# Define paths and resolve '~' to an absolute path
PROJECT_ROOT = Path("/content/CAC_2026")
SPICE_DIR = PROJECT_ROOT / "spice"
scripts_path = str(PROJECT_ROOT / "scripts")

# Add the scripts folder to the system path
if scripts_path not in sys.path:
    sys.path.append(scripts_path)

from analysis_util import find_switching_time, get_sample, apply_time_shift
from plot_framework import iplot, ioverlay, isweep, istack, isweep_overlay
from ngspice_loader import load_wrdata, load_sweep
from vcdvcd import VCDVCD

from bokeh.io import output_notebook

Get:1 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:2 https://cli.github.com/packages stable InRelease [3,917 B]
Get:3 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Hit:4 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:5 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:6 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:7 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:8 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [88.5 kB]
Get:9 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Hit:10 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Get:11 http://security.ubuntu.com/ubuntu jammy-security/universe amd64 Packages [1,311 kB]
Get:12 http://security.ubuntu.com/ubuntu jammy-security/restricted amd64 Packages [6,935 kB]
Get:13 https://r2u.stat.illinois.edu/ubuntu jammy/main all Packages [10.0 

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
100 8900k    0 8900k    0     0  10.1M      0 --:--:-- --:--:-- --:--:-- 24.2M


unpacking 'github:fossi-foundation/nix-eda/8f990fb77529c09e540e453cd836af9930ec58db?narHash=sha256-nSKBMGP8/ZC7qB3Lzd%2BFwM8REqOxlh8wpYDf2hlK6Gg%3D' into the Git cache...
copying path '/nix/store/sizirny50f893gx0gbivbqyys4fxghnq-source' from 'https://cache.nixos.org'...
unpacking 'github:numtide/devshell/255a2b1725a20d060f566e4755dbf571bbbb5f76?narHash=sha256-460jc0%2BCZfyaO8%2Bw8JNtlClB2n4ui1RbHfPTLkpwhU8%3D' into the Git cache...
copying path '/nix/store/7fd6kzwyq6pcaigmq887p9wccl7v3dl6-source' from 'https://cache.nixos.org'...
this derivation will be built:
  /nix/store/jgd55f2c85x2qc6jbynqpawn9ky5vxml-librelane-colab-env.drv
these 318 paths will be fetched (818.5 MiB download, 4.8 GiB unpacked):
  /nix/store/6rkc8a7wg89fvpvdnq2nvjk46wya6jly-abseil-cpp-20240722.1
  /nix/store/7qfvcajvjs89fxqk4379zhbdmlmxjaxb-abseil-cpp-20250814.1
  /nix/store/fwfpzqrvzhpjp91rbnq64z07diyicbwh-acl-2.3.2
  /nix/store/gmpw5fapb46sm5fxirwsyl1zicx2mrax-alsa-lib-1.2.14
  /nix/store/7l478kjr588ic8l6l32qxr60

Version 8afc8346a57fe1ab7934ba5a6056ea8b43078e71 not found locally, attempting to download…

Output()

...

Output()

Output()

Output()

Version 8afc8346a57fe1ab7934ba5a6056ea8b43078e71 enabled for the sky130 PDK.

/content
Cloning into 'CAC_2026'...
remote: Enumerating objects: 2001, done.
remote: Counting objects: 100% (111/111), done.
remote: Compressing objects: 100% (76/76), done.
remote: Total 2001 (delta 54), reused 79 (delta 35), pack-reused 1890 (from 2)
Receiving objects: 100% (2001/2001), 19.84 MiB | 22.67 MiB/s, done.
Resolving deltas: 100% (595/595), done.
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
Suggested packages:
  gtkwave
The following NEW packages will be installed:
  iverilog
0 upgraded, 1 newly installed, 0 to remove and 28 not upgraded.
Need to get 2,130 kB of archives.
After this operation, 6,749 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/universe amd64 iverilog amd64 11.0-1.1 [2,130 kB]
Fetched 2,130 kB in 1s (2,170 kB/s)
debconf: unable to initialize frontend: Dialog
debconf: (No usable dialog-like program is installed, so the dialog based frontend cannot be used. at /usr/sh

## What is a Delay-Locked Loop
---

A Delay-locked Loop (DLL) is a closed-loop feedback circuit that aligns the phase of an output clock to a reference clock. It shifts the phase of this output clock by delaying it, either through a chain of discrete digital delay cells or through a voltage-controlled analog delay line, until the output edge lines up with the reference edge. Unlike a PLL, a DLL does not generate a new frequency.

The DLL's core feedback loop consists of three blocks: a Phase Detector, a Controller, and a delay line, for which this notebook uses a Digitally Controlled Delay Line (DCDL). Together, they form a feedback loop that continuously adjusts the delay applied to the output clock until its phase matches the reference, at which point, the system is "locked."

> **TODO: Add draw.io diagram (top-level DLL block diagram: CLK_REF -> Phase Detector -> Controller -> DCDL -> CLK_OUT feedback loop)**

Each module section below describes the high-level function, presents each implementation variant with its pros and cons, as evidenced by the spice simulations.

## Phase Detector
---

The phase detector compares the rising edges of the reference clock (`clk_in`) and the feedback clock (`clk_out`) and produces two single-bit outputs: `up` (reference leads, increase the delay) and `down` (feedback leads, decrease the delay). It is important to note that the phase detector does not measure the magnitude of the phase error, only the direction.

A bang-bang style of detection is simple, but it means the loop can never settle perfectly still. Near lock, the detector continuously toggles between `up` and `down`, producing jitter. Other phase detector architectures, as explored below, can avoid this intrinsic toggling, but at the cost of added complexity. The choice of phase detector architecture directly impacts the loop's accuracy, stability, and susceptibility to metastability.

---
### Implementation 1: Single Flip-Flop Detector

<p align="center">
  <img src="https://github.com/SiliconJackets/CaC_Spring26/blob/main/images/pd/pd.png?raw=true" width="500">
</p>

This is perhaps the most trivial phase detector. A single D type flip-flop samples `clk_out` on the rising edge of `clk_in`. If `clk_out` is low at the sampling instant, the feedback clock is trailing behind the reference clock, hence `up` is asserted. Conversely, `down` is asserted if `clk_out` is high at the sampling instant.

There is no "equal" state; the detector always picks a direction.

| Pros | Cons |
|----------|----------|
| Extremely low area and complexity. A single flip-flop plus output decode. Smallest power and area footprint of all detectors. | Inherently biased. When clocks are perfectly aligned, the detector still asserts `down` because `clk_out` is sampled as high at the rising edge of `clk_in`. The bias means the loop locks with a small systematic phase offset rather than true zero error. |
| Fully synthesizable, fast timing. Minimal critical path yields high maximum operating frequency (best performance). | No frequency detection capability. |
| | Persistent bang-bang toggling near lock limits achievable jitter performance. |


In [14]:
# @markdown
# @markdown Click the ▷ button to run the flow for a Single Flip-Flop Detector (neccesary if you want to run custom spice simulations)
# @markdown

!python3 CAC_2026/scripts/flow_cli.py phase_detector_syn_ff1

-> Starting flow for 'phase_detector_syn_ff1'...
/content/CAC_2026/librelane/design/phase_detector_syn_ff1/config.json
──────────────────────────────── Verilator Lint ────────────────────────────────
────────────────────────── Lint Timing Errors Checker ──────────────────────────
───────────────────────────── Lint Errors Checker ──────────────────────────────
──────────────────────────── Lint Warnings Checker ─────────────────────────────
───────────────────────────── Generate JSON Header ─────────────────────────────
────────────────────────────────── Synthesis ───────────────────────────────────
──────────────────────────── Unmapped Cells Checker ────────────────────────────
────────────────────────────── Yosys Synth Checks ──────────────────────────────
─────────────────────── Netlist Assign Statement Checker ───────────────────────
─────────────────────────────── Check SDC Files ────────────────────────────────
──────────────────────────── Check Macro Instances ────────────────────

In [20]:
# @markdown Change these parameters to run your own post-synthesis simulations for this phase detector (NOTE: Each spice simulation will take some time to run)

clk_in_delay = 20  # @param {key:"CLK_IN Delay From Start", type:"integer"}
clk_out_delay = 40  # @param {key:"CLK_OUT Delay From Start", type:"integer"}

sim_output = !python CAC_2026/scripts/flow_cli.py phase_detector_syn_ff1 --process spice --clk-in-delay {clk_in_delay}n --clk-out-delay {clk_out_delay}n

file_name = f"phase_detector_syn_ff1_clkin{clk_in_delay}n_clkout{clk_out_delay}n.csv"

traces = load_wrdata(
    SPICE_DIR / "tmp_results" / file_name,
    ["CLK_IN", "CLK_OUT", "RST", "UP", "DOWN"],
)

istack(
    layers=[
        {"CLK_IN": traces["CLK_IN"], "CLK_OUT": traces["CLK_OUT"]},
        {"RST": traces["RST"]},
        {"UP": traces["UP"], "DOWN": traces["DOWN"]},
    ],
    title=f"FF1 phase detector — {clk_in_delay}ns delay for clk_in and {clk_out_delay}ns delay for clk_out",
    xlabel="Time (ns)",
    ylabels=["Clocks (V)", "Reset (V)", "Outputs (V)"],
)

In [25]:
# @markdown If you don't want to wait for spice simulations, view our previously ran results here:

traces = load_wrdata(
    SPICE_DIR / "results" / "pd_syn_ff1_clkout_lead_results.csv",
    ["CLK_IN", "CLK_OUT", "RST", "UP", "DOWN"],
)

istack(
    layers=[
        {"CLK_IN": traces["CLK_IN"], "CLK_OUT": traces["CLK_OUT"]},
        {"RST": traces["RST"]},
        {"UP": traces["UP"], "DOWN": traces["DOWN"]},
    ],
    title="FF1 phase detector — clk_out leads",
    xlabel="Time (ns)",
    ylabels=["Clocks (V)", "Reset (V)", "Outputs (V)"],
)

traces = load_wrdata(
    SPICE_DIR / "results" / "pd_syn_ff1_clkin_lead_results.csv",
    ["CLK_IN", "CLK_OUT", "RST", "UP", "DOWN"],
)

istack(
    layers=[
        {"CLK_IN": traces["CLK_IN"], "CLK_OUT": traces["CLK_OUT"]},
        {"RST": traces["RST"]},
        {"UP": traces["UP"], "DOWN": traces["DOWN"]},
    ],
    title="FF1 phase detector — clk_in leads",
    xlabel="Time (ns)",
    ylabels=["Clocks (V)", "Reset (V)", "Outputs (V)"],
)


> **TODO: Add data analysis (bias characterization, jitter measurements)**

---

### Implementation 2: Edge-Order Detector

<p align="center">
  <img src="https://github.com/SiliconJackets/CaC_Spring26/blob/main/images/pd/edgeorderpd.png?raw=true" width="500">
</p>

This is an extension of the single flip-flop detector. Each clock edge tries to assert its corresponding output (`up` or `down`), but only succeeds if the other clock's edge has not already arrived. When `clk_in` rises first, `up` is set; when `clk_out` rises first, `down` is set. The opposite edge clears the output, completing the comparison for that cycle.

| Pros | Cons |
|----------|----------|
| Minimal logic. Two flip-flop-style blocks with cross-coupled gating. Low area and power, only marginally more than the single-FF detector. | Race conditions when edges arrive nearly simultaneously; both `up` and `down` can briefly assert (observed in aligned-edge test cases). |
| No explicit reset feedback path needed. | When `clk_out` leads `clk_in`, the detector can still incorrectly assert `up` because the first-edge-wins logic lets `clk_in` capture before `down` registers the earlier feedback edge. |
| | No frequency detection capability. Performance is limited by metastability susceptibility when edges are close. |


In [22]:
# @markdown
# @markdown Click the ▷ button to run the flow for a Edge-Order Detector (neccesary if you want to run custom spice simulations)
# @markdown

!python3 CAC_2026/scripts/flow_cli.py phase_detector_syn_edge

-> Starting flow for 'phase_detector_syn_edge'...
/content/CAC_2026/librelane/design/phase_detector_syn_edge/config.json
──────────────────────────────── Verilator Lint ────────────────────────────────
────────────────────────── Lint Timing Errors Checker ──────────────────────────
───────────────────────────── Lint Errors Checker ──────────────────────────────
──────────────────────────── Lint Warnings Checker ─────────────────────────────
───────────────────────────── Generate JSON Header ─────────────────────────────
────────────────────────────────── Synthesis ───────────────────────────────────
──────────────────────────── Unmapped Cells Checker ────────────────────────────
────────────────────────────── Yosys Synth Checks ──────────────────────────────
─────────────────────── Netlist Assign Statement Checker ───────────────────────
─────────────────────────────── Check SDC Files ────────────────────────────────
──────────────────────────── Check Macro Instances ──────────────────

In [23]:
# @markdown Change these parameters to run your own post-synthesis simulations for this phase detector (NOTE: Each spice simulation will take some time to run)

clk_in_delay = 20  # @param {key:"CLK_IN Delay From Start", type:"integer"}
clk_out_delay = 25  # @param {key:"CLK_OUT Delay From Start", type:"integer"}

sim_output = !python CAC_2026/scripts/flow_cli.py phase_detector_syn_edge --process spice --clk-in-delay {clk_in_delay}n --clk-out-delay {clk_out_delay}n

file_name = f"phase_detector_syn_edge_clkin{clk_in_delay}n_clkout{clk_out_delay}n.csv"

traces = load_wrdata(
    SPICE_DIR / "tmp_results" / file_name,
    ["CLK_IN", "CLK_OUT", "RST", "UP", "DOWN"],
)

istack(
    layers=[
        {"CLK_IN": traces["CLK_IN"], "CLK_OUT": traces["CLK_OUT"]},
        {"RST": traces["RST"]},
        {"UP": traces["UP"], "DOWN": traces["DOWN"]},
    ],
    title=f"Edge phase detector — {clk_in_delay}ns delay for clk_in and {clk_out_delay}ns delay for clk_out",
    xlabel="Time (ns)",
    ylabels=["Clocks (V)", "Reset (V)", "Outputs (V)"],
)

In [24]:
# @markdown If you don't want to wait for spice simulations, view our previously ran results here:

traces = load_wrdata(
    SPICE_DIR / "results" / "pd_syn_edge_clkout_lead_results.csv",
    ["CLK_IN", "CLK_OUT", "RST", "UP", "DOWN"],
)

istack(
    layers=[
        {"CLK_IN": traces["CLK_IN"], "CLK_OUT": traces["CLK_OUT"]},
        {"RST": traces["RST"]},
        {"UP": traces["UP"], "DOWN": traces["DOWN"]},
    ],
    title="FF1 phase detector — clk_out leads",
    xlabel="Time (ns)",
    ylabels=["Clocks (V)", "Reset (V)", "Outputs (V)"],
)

traces = load_wrdata(
    SPICE_DIR / "results" / "pd_syn_edge_clkin_lead_results.csv",
    ["CLK_IN", "CLK_OUT", "RST", "UP", "DOWN"],
)

istack(
    layers=[
        {"CLK_IN": traces["CLK_IN"], "CLK_OUT": traces["CLK_OUT"]},
        {"RST": traces["RST"]},
        {"UP": traces["UP"], "DOWN": traces["DOWN"]},
    ],
    title="FF1 phase detector — clk_in leads",
    xlabel="Time (ns)",
    ylabels=["Clocks (V)", "Reset (V)", "Outputs (V)"],
)


> **TODO: Add simulation results / waveform figures for edge-order detector**

> **TODO: Add data analysis (measured accuracy across different phase offsets)**

---



### Implementation 3: Phase-Frequency Detector (PFD)

<p align="center">
  <img src="https://github.com/SiliconJackets/CaC_Spring26/blob/main/images/pd/pfdpd.png?raw=true" width="500">
</p>

This is the industry-standard architecture. The analysis below will prove why. Two flip-flops are held with D=1. The rising edge of `clk_in` sets the first FF (asserting `up`), and the rising edge of `clk_out` sets the second FF (asserting `down`). An AND gate generates a reset pulse that clears both flip-flops when both outputs are simultaneously high. The width of the `up` or `down` pulse is proportional to the phase difference between the two clocks. This implementation can also detect frequency mismatches; if one clock is faster, its corresponding output will be asserted for longer durations, on average, driving the loop in the correct direction. It is important to note that this implementation is well suited for PLLs, the frequency detection capability is unnecessary in a DLL. Only the phase-detection behavior is used.

| Pros | Cons |
|----------|----------|
| Detects both phase error and frequency error. The only detector here with frequency acquisition capability. | Dead zone. When phase difference is very small, the reset pulse fires almost immediately, producing narrow output pulses that may not be captured reliably by downstream logic. |
| Pulse width encodes error magnitude (useful for analog loops, and provides clean direction for digital bang-bang use). | Slightly more area and power than single-FF or edge-order approaches (two flip-flops, AND gate, and reset feedback path). |
| Well-understood, widely used in PLL/DLL designs. Robust and predictable behavior across PVT corners. | The AND-gate reset path introduces a minimum pulse width constraint that can limit maximum operating frequency. |

In [ ]:
!python3 CAC_2026/scripts/flow_cli.py phase_detector_syn_pfd



> **TODO: Add simulation results / waveform figures for PFD**

> **TODO: Add data analysis (dead zone characterization, frequency acquisition behavior)**

---




### Implementation 4: Sampled XOR Detector

<p align="center">
  <img src="https://github.com/SiliconJackets/CaC_Spring26/blob/main/images/pd/sampledxorpd.png?raw=true" width="500">
</p>

An XOR gate compares the two clocks. If the signals mismatch, `1` will be produced. This signal is then sampled on the rising edge of teh reference clock to determine direction. If the feedback clock is low at the sampling instant, the reference is leading (`up`); if the feedback clock is high, the feedback is leading (`down`); if b oth clocks agree, both outputs are cleared.

However, a fundamental problem with this architecture is that the XOR gate and the direction sample are evaluated at the same instant, the reference clock's rising edge. Reference clock is always high at this moment. The XOR reduces to simply checking whether the feedback clock is low or high. When the feedback clock is low, the XOR gate sees a mismatch and `up` is asserted. When the feedback clock is high, the XOR gate sees no mismatch and both outputs are cleared. The `down` path is never reached. **The detector can only increase delay or hold; it cannot decrease delay.** This makes the loop fundamentally one-directional.

| Pros | Cons |
|----------|----------|
| Simple combinational front-end (single XOR gate). Minimal area overhead for the detection logic. | One-directional: the detector can only assert `up` or clear both outputs. It cannot assert `down`, so the DLL loop can only increase delay, never decrease it. |
| Naturally produces a "no error" state when clocks are aligned (both outputs cleared). | XOR detects mismatch but not direction; direction is inferred by sampling the feedback clock level, making it duty-cycle and timing dependent. |
| Low power due to minimal switching activity (one XOR gate, one flip-flop). Smallest dynamic power of all detectors. | Not suitable as a standalone detector in practical designs due to the single-sided output limitation. |

In [3]:
!python3 CAC_2026/scripts/flow_cli.py phase_detector_syn_xor1

-> Starting flow for 'phase_detector_syn_xor1'...
/content/CAC_2026/librelane/design/phase_detector_syn_xor1/config.json
──────────────────────────────── Verilator Lint ────────────────────────────────
────────────────────────── Lint Timing Errors Checker ──────────────────────────
───────────────────────────── Lint Errors Checker ──────────────────────────────
──────────────────────────── Lint Warnings Checker ─────────────────────────────
───────────────────────────── Generate JSON Header ─────────────────────────────
────────────────────────────────── Synthesis ───────────────────────────────────
──────────────────────────── Unmapped Cells Checker ────────────────────────────
────────────────────────────── Yosys Synth Checks ──────────────────────────────
─────────────────────── Netlist Assign Statement Checker ───────────────────────
─────────────────────────────── Check SDC Files ────────────────────────────────
──────────────────────────── Check Macro Instances ──────────────────

In [6]:
# @markdown
# @markdown Click the ▷ button to graph the functional waveform of how the XOR Phase Detector should work
# @markdown

design_file = "/content/CAC_2026/design_root/src/verilog/parts/phase_detector/phase_detector_syn_xor1.sv"
testbench_file = "/content/CAC_2026/design_root/src/verilog/parts/phase_detector/tb_phase_detector_syn_xor1.sv"

!iverilog -o simulation.vvp {design_file} {testbench_file}

!vvp simulation.vvp

output_notebook()  # Initialize Bokeh to display inline in the Colab notebook

vcd_file_path = '/content/tb_phase_detector_syn_xor1.vcd'

if not os.path.exists(vcd_file_path):
    print(f"Error: Could not find '{vcd_file_path}'. Make sure it is uploaded to Colab.")
else:
    vcd = VCDVCD(vcd_file_path)

    end_time = 0
    for sig in vcd.signals:
        if vcd[sig].tv:
            last_event_time = vcd[sig].tv[-1][0]
            if last_event_time > end_time:
                end_time = last_event_time

    def get_vcd_trace(sig_name):
        if sig_name not in vcd.signals:
            print(f"Warning: {sig_name} not found in VCD.")
            return ([0, end_time], [0, 0])

        signal_data = vcd[sig_name]
        times = []
        values = []
        for time, val_str in signal_data.tv:
            times.append(time)

            if val_str.lower() in ('x', 'z'):
                val = 0.5
            elif val_str.startswith('b'):
                try: val = int(val_str[1:], 2)
                except ValueError: val = 0
            else:
                try: val = int(val_str)
                except ValueError: val = 0
            values.append(val)

        if times:
            times.append(end_time)
            values.append(values[-1])
        else:
            times = [0, end_time]
            values = [0, 0]

        return (times, values)

    prefix = "tb_phase_detector.dut."
    traces = {
        "CLK_IN": get_vcd_trace(f"{prefix}clk_in"),
        "CLK_OUT": get_vcd_trace(f"{prefix}clk_out"),
        "RST": get_vcd_trace(f"{prefix}rst"),
        "PHASE_ERROR": get_vcd_trace(f"{prefix}phase_error"),
        "UP": get_vcd_trace(f"{prefix}up"),
        "DOWN": get_vcd_trace(f"{prefix}down"),
    }

    istack(
        layers=[
            {"CLK_IN": traces["CLK_IN"], "CLK_OUT": traces["CLK_OUT"]},
            {"RST": traces["RST"]},
            {"UP": traces["UP"], "DOWN": traces["DOWN"]},
        ],
        title="Verilog FF1 Phase Detector — clk_out leads",
        xlabel="Time (ps)",   # Based on standard `timescale 1ps/1ps
        ylabels=["Clocks (Logic)", "Reset", "Phase Err", "Outputs"],
        kind="step",          # Forces square digital waveforms
        layer_height=150,     # Adjusted height for 4 layers
    )

VCD info: dumpfile tb_phase_detector_syn_xor1.vcd opened for output.
Simulation complete.
/content/CAC_2026/design_root/src/verilog/parts/phase_detector/tb_phase_detector_syn_xor1.sv:46: $finish called at 100000 (1ps)


> **TODO: Add simulation results / waveform figures for XOR detector**

> **TODO: Add data analysis (duty cycle sensitivity, directional bias measurements)**

## Controller
---

The controller recieves the `up`/`down` signals from the phase detector and maintains an N-bit control word that sets the delay through the DCDL. At a high level, every controller variant is an accumulator. When `up` is asserted, the control word increments (more delay), when `down` is asserted, the control world decrements (less delay), and when both match, it holds.

The difference between the implementions below is how aggressively they update. The step size, filtering, and mode-switching strategies each trade off lock speed against steady-state jitter. The phase detector provides direction, the controller sets the "bandwidth". Large steps mean fast convergence, but more overshoot. Small steps more smooth tracking, but slower acquisition.

---

### Implementation 1: Saturating Controller

<p align="center">
  <img src="https://github.com/SiliconJackets/CaC_Spring26/blob/main/images/controller/saturating.png?raw=true" width="500">
</p>


This is the trivial, baseline design. The implementation is the high level overview mentioned above. On each clock edge, the control world increments or decrements by 1 depending on if `up` or `down` is asserted. Hard saturation clamps `ctrl` at 0 and `2^N - 1` to prevent underflow or overflow. On reset, the control world is initialized to a parameterized `INIT_CTRL` value (default midpoint of 32 for 6-bit control).

| Pros | Cons |
|----------|----------|
| Simplest possible controller. Easy to verify and reason about. | Fixed +-1 step means slow convergence when far from lock. |
| Predictable, monotonic response to persistent error. | Near lock, the +-1 toggling produces a limit cycle with 1-LSB jitter that cannot be reduced without external filtering. |
| Smallest area and lowest power of all controllers: one adder/subtractor, one comparator, one register. Industry baseline for comparison. | No adaptability. Performance is the same whether the loop is far from or near lock. |

In [ ]:
!python3 CAC_2026/scripts/flow_cli.py TODO



> **TODO: Add simulation results / waveform figures for saturating controller**

> **TODO: Add data analysis (lock time, steady-state jitter amplitude)**

---

### Implementation 2: Filtered Controller

<p align="center">
  <img src="https://github.com/SiliconJackets/CaC_Spring26/blob/main/images/controller/filter.png?raw=true" width="500">
</p>


Instead of updating the control word on every cycle, this controller requires `FILTER_LEN` consecutive `up` (or `down`) requests before incrementing or decrementing by 1. Two internal counters track consecutive `up` and `down` assertions. If the direction changes or both signals are idle, both counters reset to zero. Only when one counter reaches the threshold does the control word update, and then that counter resets. This acts as a simple digital low-pass filter on the phase detector output.

| Pros | Cons |
|----------|----------|
| Significantly reduces jitter near lock; transient noise or single-cycle glitches from the phase detector are ignored. | Slower lock acquisition. The loop must see `FILTER_LEN` consistent cycles before taking any action. |
| Steady-state limit cycle is suppressed because the toggling `up`/`down` pattern resets the counters before threshold is reached. | The `FILTER_LEN` parameter requires tuning: too small and it behaves like the saturating controller, too large and the loop becomes unresponsive. |
| Modest area increase over the saturating controller (two additional counters and comparators). Power consumption is slightly higher due to counter activity, but reduced output toggling saves downstream switching power. | Effective loop bandwidth is reduced by the filter factor, which can be problematic if the clock environment changes rapidly. |

In [ ]:
!python3 CAC_2026/scripts/flow_cli.py TODO



> **TODO: Add simulation results / waveform figures for filtered controller**

> **TODO: Add data analysis (lock time vs FILTER_LEN, jitter reduction compared to saturating)**

---

### Implementation 3: Acquire/Track Controller

<p align="center">
  <img src="https://github.com/SiliconJackets/CaC_Spring26/blob/main/images/controller/aquiretrack.png?raw=true" width="500">
</p>

The acquire/track controller operates in two modes to balance a fast lock acquisition against low steady-state jitter. On startup, it is in aquire mode and adjusts the delay by a coarse step (`ACQUIRE_STEP`, default 4) on every `up`/`down` assertion from the phase detector. This allows the DLL to converge rapidly towards the lock point. A quiet-cycle counter monitors phase-detector activity. Each cycle without an `up` or `down` assertion increments a counter, and any new assertion clears it. When this counter reaches the threshold `QUIET_CYCLES` (default 8), the loop is considered near lock and the controller goes into track mode. Here, the corrections uses a small step size (`TRACK_STEP`, default 1) to minimize jitter. The mode transition is one-way and held until reset, so transient disturbances after lock cannot kick the loop back into coarse-step behavior.

| Pros | Cons |
|----------|----------|
| Fast acquisition (large steps cover the control range quickly) with low steady-state jitter (small steps near lock). | One-way mode switch means the controller cannot re-acquire if the loop is disturbed after locking (e.g., supply noise kicks the delay far from lock). |
| Widely used in industry DLL/PLL designs as a practical balance of speed and stability. | The `QUIET_CYCLES` threshold must be tuned; too aggressive and the controller switches to track mode before actually reaching lock. |
| Moderate area and power - adds only a mode register, quiet counter, and step-size mux over the saturating baseline. | Larger step sizes during acquire mode can cause overshoot, especially if `ACQUIRE_STEP` is too large relative to the DCDL resolution. |

In [ ]:
!python3 CAC_2026/scripts/flow_cli.py TODO



> **TODO: Add simulation results / waveform figures for acquire/track controller**

> **TODO: Add data analysis (acquisition phase duration, jitter comparison between modes)**

---

### Implementation 4: Coarse/Fine Controller

<p align="center">
  <img src="https://github.com/SiliconJackets/CaC_Spring26/blob/main/images/controller/coarsefine.png?raw=true" width="500">
</p>

The controller partitions its control word into two concatenated fields `ctrl = {coarse, fine}`. On startup it opereates in course mode, where each `up`/`down` assertion increments or decrements only the course field. This allows the DLL to converge rapidly towards the lock point. A quiet-cycle counter monitors phase-detector activity. When this counter reaches the threshold `SWITCH_QUIET` (default 8), the loop is considered near lock and the controller transitions to fine mode. Subsequence corrections adjust only the fine field. Each field saturates independently at its own bounds, preventing carries between fields and cleanly decoupling the two adjustment ranges.

| Pros | Cons |
|----------|----------|
| High total resolution without needing an extremely wide control word; coarse bits provide range, fine bits provide precision. | More complex than a single-counter design - two independent saturating counters plus mode logic. Larger area than the saturating or filtered controllers. |
| Clean separation of acquisition (coarse) and tracking (fine) phases. | Coarse-to-fine boundary can introduce a jump if the coarse step doesn't land near the correct fine range. |
| Hardware-efficient: the two counters are small and independent. Power overhead is modest since only one counter is active at a time. | Same one-way mode switch limitation as acquire/track - cannot re-acquire after locking. |

It can be noted that acquire/track and coarse/fine are two implementations of the same dual-speed control strategy. It only differs in whether the two speeds come from changing the step size or from changing which bits of the control word are being adjusted.

In [ ]:
!python3 CAC_2026/scripts/flow_cli.py controller_2mode



> **TODO: Add simulation results / waveform figures for coarse/fine controller**

> **TODO: Add data analysis (resolution improvement, coarse-to-fine transition behavior)**

---

### Implementation 5: Variable-Step Controller

<p align="center">
  <img src="https://github.com/SiliconJackets/CaC_Spring26/blob/main/images/controller/variablestep.png?raw=true" width="500">
</p>

This controller adapts its step size based on the consecutive `up`/`down` assertions. There exists a counter, and each repeat increments the counter, while any direction reversal resets it to 1. The step size is then selected from the counter value in three tiers, a unit step of 1 by default, growing to `MED_STEP` (default 2) once the count exceeds `MED_THRESH` (default 4), and jumping again to BIG_STEP (default 4) once it exceeds `BIG_THRESH` (default 8). Persistent same-direction error is interpreted the loop is far from the lock point, so the controller "sprints" towards it with progressively larger corrections. As soon as the phase detector begins alternating firections, the counter collapses and the loop reverts to fine adjustments.

| Pros | Cons |
|----------|----------|
| Fastest convergence of all five controllers - accelerates when far from lock without requiring an explicit mode switch. | More tuning parameters (`MED_THRESH`, `BIG_THRESH`, `MED_STEP`, `BIG_STEP`); poor tuning can cause overshoot or oscillation. |
| Naturally adapts to both acquisition and tracking without separate modes. Can re-accelerate if disturbed, unlike the one-way mode controllers. | Near lock, even a short run of consistent `up` or `down` from the limit cycle can trigger medium steps, potentially increasing jitter compared to a fixed +-1 controller. |
| Moderate area cost - adds a direction register, 4-bit counter, and step-size comparators/mux. Power scales with activity since larger steps mean fewer total update cycles to reach lock. | Most complex controller to verify. Non-linear step behavior makes worst-case jitter harder to bound analytically. |


In [ ]:
!python3 CAC_2026/scripts/flow_cli.py controller_variable_step


> **TODO: Add simulation results / waveform figures for variable-step controller**

> **TODO: Add data analysis (convergence speed comparison, overshoot characterization)**

## DCDL (Digitally Controlled Delay Line)
---

The DCDL is the actuator of the DLL. It receives the digital control word from the controller and converts it into a physical propagation delay applied to the clock signal. The clock enters a chain of delay elements, and the control word determines how many stages the signal passes through before reaching the output. A larger control value means more active delay stages and a longer delay.

The DCDL must provide monotonic, glitch-free delay adjustment across its full control range. What varies between the implementations below is the delay primitive (inverters vs. NAND gates), the tap selection strategy (mux trees vs. bypass logic), and how each deals with signal polarity. They span from simple inverter chains to dual-chain vernier interpolation.

---

### Implementation 1: Inverter-Based DCDL

<p align="center">
  <img src="https://github.com/SiliconJackets/CaC_Spring26/blob/main/images/dcdl/4stageinverter.png?raw=true" width="500">
</p>

Each delay stage consists of two inverters in series, which preserves signal polarity (an even number of inversions). The input propagates through the chain, producing one tap per stage. A hierarchical mux tree, controlled by the control word, selects which tap is routed to the output. This implementation uses a chain of 4 double-inverter stages with a 2-bit control word.

| Pros | Cons |
|----------|----------|
| Straightforward design; the double-inverter stages guarantee consistent polarity at every tap. | Two inverters per stage means more area and power per delay step compared to single-inverter designs. |
| Hierarchical mux tree scales cleanly with more stages. | Mux tree adds its own propagation delay to every path, which may limit minimum achievable delay. |
| Easy to reason about delay linearity. Good baseline for area vs. resolution comparison. | Coarse resolution. Extending to more taps requires a wider chain and deeper mux tree, increasing area and power. |

In [ ]:
!python3 CAC_2026/scripts/flow_cli.py inv_dcdl



> **TODO: Add simulation results / waveform figures for inverter-based DCDL**

> **TODO: Add data analysis (delay per step, linearity measurement)**

---

### Implementation 2: Conditional Inverter DCDL

<p align="center">
  <img src="https://github.com/SiliconJackets/CaC_Spring26/blob/main/images/dcdl/4stageconditional.png?raw=true" width="500">
</p>

This design uses a single inverter per stage instead of two, halving the chain length for the same number of taps. The trade-off is that odd-numbered taps produce an inverted signal (odd number of inversions from the input), while even-numbered taps are non-inverted. A mux tree selects the desired tap, and an XNOR gate at the output conditionally flips the polarity based on a control bit.

However, the polarity correction is imperfect in this implementation. The XNOR keys on one bit of the control word, but the actual inversion parity of the selected tap depends on a different bit. The correction only produces the correct output polarity for half of the control word values; for the other half, the output is inverted.

| Pros | Cons |
|----------|----------|
| Half the inverters of the double-inverter design for the same number of taps. Significantly smaller area and lower power. | The XNOR correction gate adds a fixed delay to every output path, and its own delay variation can affect linearity. |
| Finer delay granularity per stage (each inverter adds one gate delay instead of two). | Polarity correction is only valid for half the control word values. The remaining values produce an inverted output. |
| Lower power consumption due to fewer switching elements. | Extending to wider control words requires careful bookkeeping to maintain correct polarity across all tap selections. |

In [ ]:
!python3 CAC_2026/scripts/flow_cli.py inv_dcdl_cond


> **TODO: Add simulation results / additional waveform figures for conditional inverter DCDL**

> **TODO: Add data analysis (delay per step vs. Implementation 1, area comparison)**

---

### Implementation 3: Glitch-Free Inverter DCDL

<p align="center">
  <img src="https://github.com/SiliconJackets/CaC_Spring26/blob/main/images/dcdl/glitchfree.png?raw=true" width="500">
</p>

This variant addresses glitch hazards that occur in the mux-based designs when the select signal changes while the clock is propagating through the chain. The approach replaces the mux tree with a NAND-based gating and combining scheme. The control word is decoded into a one-hot signal and registered on the clock edge, so the active tap can only change synchronously. Each tap is inverted and gated with its one-hot select bit through a NAND gate, and the gated outputs are combined through a NAND tree to produce the final output.

The first tap is wired directly to the input (zero inverter delay), and each subsequent tap adds one inverter delay. On reset, the registered select defaults to the first tap.

| Pros | Cons |
|----------|----------|
| Glitch-free by construction; the registered one-hot select ensures no transient paths are created during control word changes. | Requires a clock and reset input, unlike the purely combinational implementations above. Adds sequential area for the registered select. |
| NAND-based gating and combining is area-efficient in standard cell libraries. | The registered select adds one cycle of latency between a control word change and the delay actually updating. This impacts loop response time. |
| Eliminates potential timing violations caused by mid-transition glitches in the other DCDL variants. | The NAND tree path adds fixed delay overhead. More total gates than the simple mux-tree approaches, increasing area and static power. |

In [ ]:
!python3 CAC_2026/scripts/flow_cli.py TODO



> **TODO: Add simulation results / additional waveform figures for glitch-free DCDL**

> **TODO: Add data analysis (glitch characterization vs. Implementations 1 and 2)**

---

### Implementation 4: NAND-Based DCDL

<p align="center">
  <img src="https://github.com/SiliconJackets/CaC_Spring26/blob/main/images/dcdl/nand.png?raw=true" width="500">
</p>

A fundamentally different approach. Instead of selecting a tap from a shared delay chain, each stage is a NAND-based cell that can either propagate the signal from the previous stage (adding delay) or bypass it by injecting the input signal directly. Each cell's control bit determines which path is active.

The cells are chained in series. The head of the chain is initialized to a static value, so it cannot pass a valid signal on its own. The first cell whose control bit enables bypass injects the input clock into the chain. From that point onward, cells with bypass disabled propagate the signal through their NAND logic, each adding gate delay. The effective delay scales with how many propagation stages sit between the injection point and the output. This implementation uses 4 cells with a 4-bit control word. Synthesis annotations prevent the tool from optimizing away the intentional delay structure.

| Pros | Cons |
|----------|----------|
| Wider delay range per stage than inverter-based designs, since each NAND cell introduces more gate delay than a single inverter. | Non-linear delay steps; the delay contribution of each cell depends on its position and the state of other cells. |
| Synthesis-robust: annotations preserve the intentional delay structure through the tool flow. | More complex per-cell logic (three NAND gates and an inverter per stage). Higher area and power per stage than inverter-based designs. |
| Wider control word gives more delay range than the inverter-based DCDLs. | Requires priority-based control encoding. The first enabled bypass from the head of the chain determines the injection point; arbitrary bit patterns may produce unexpected outputs or stuck-at-zero. |

In [ ]:
!python3 CAC_2026/scripts/flow_cli.py nand_dcdl



> **TODO: Add simulation results / additional waveform figures for NAND-based DCDL**

> **TODO: Add data analysis (delay range, per-step nonlinearity)**

---

### Implementation 5: Vernier Crossover DCDL

> **TODO: Add draw.io diagram (dual chain: slow buf_1 chain and fast buf_4 chain with crossover muxes at each stage controlled by Q[i], output from fast_net[STAGES])**

The most sophisticated DCDL architecture here, targeting fine delay resolution through interpolation between two parallel buffer chains with different per-stage delays. A "slow" chain is built from small, high-delay buffers and a "fast" chain from large, low-delay buffers. At each stage, a mux can cross the signal from the slow chain into the fast chain. The input enters the slow chain; the fast chain is initialized to a static value.

The control word determines where the crossover happens. When a control bit is asserted at a given stage, the slow-chain signal crosses into the fast chain and propagates through the remaining fast-chain stages to the output. More slow stages before the crossover means more total delay. The achievable delay step is the difference between one slow-buffer delay and one fast-buffer delay, which can be much smaller than either buffer's absolute delay. This implementation uses 16 stages with `sky130_fd_sc_hd__buf_1` (slow) and `sky130_fd_sc_hd__buf_4` (fast) standard cells.

| Pros | Cons |
|----------|----------|
| Very fine delay resolution; the delay step is the difference between the slow and fast buffer delays, enabling sub-gate-delay granularity. Best resolution of all DCDL variants. | Largest area and power of all five implementations. Two full buffer chains plus a mux per stage. Highest static and dynamic power. |
| Many stages with per-stage control gives high granularity. | Delay linearity depends on matching between the slow and fast cell delays across PVT corners. |
| Uses sky130 standard cells directly. Ready for physical implementation with no custom cell design. | The fast chain is initialized to a static value, so the output is invalid until at least one crossover mux is enabled. Both chains are always active, wasting power on the unused path. |


In [ ]:
!python3 CAC_2026/scripts/flow_cli.py vernier_dcdl



> **TODO: Add simulation results / waveform figures for vernier DCDL**

> **TODO: Add data analysis (delay resolution measurement, linearity across PVT corners)**

# Final Architecture Selection and System Integration

---

The concept of Delay Locked Loops (DLLs) have several real and specialized hardware implementations that are visible across modern digital systems. After exploring multiple implementations of the Phase Detector (PD), Controller, and Digitally Controlled Delay Line (DCDL), the next step was to realize different combinations of these components into complete architectures. The final implementations were selected with an emphasis on realistic hardware use cases and physical design constraints rather than purely theoretical performance.

Rather than optimizing for a single metric such as area or speed, the design process prioritized a system-level balance between:

- Deterministic convergence behavior
- Robustness under process, voltage, and temperature (PVT) variations
- Synthesizability within the chosen flow (Sky130)
- Clean separation between control logic and delay line

The major objective of this section is not only to demonstrate functional correctness, but to ensure that each architecture reflects realistic hardware behavior, where timing uncertainty, glitches, and post-layout effects significantly impact system performance.


---

## System-Level vs Circuit-Level Perspective

While DLLs are often described using high-level block diagrams, their true behavior emerges only after physical implementation. This requires considering both system-level and circuit-level effects simultaneously.
### System-Level Considerations

- Lock time
- Convergence stability
- Sensitivity to PVT variations
- Phase error behavior

### Circuit-Level Realities

- Gate delay variation across standard cells
- Fanout loading in delay chains
- Clock-to-Q delays in flip-flops (affecting PFD timing)
- Glitch propagation due to combinational hazards
- Routing-induced skew and parasitic

This distinction is critical because architectures that appear equivalent in simulation often diverge significantly after layout. As a result, the design approach adopted here explicitly considers both levels simultaneously, ensuring that architectural decisions remain valid under realistic implementation conditions.

---

## Architectural Exploration

Instead of converging on a single “optimal” design, three fundamentally different DLL architectures were explored, each representing a distinct interpretation of timing control:

- **Zero-Delay Buffer (ZDB):** Delay cancellation through feedback
- **Multiphase DLL:** Phase generation using delay taps
- **TDC-Based DLL:** Time measurement via delay encoding

These perspectives are intentionally orthogonal. While most designs emphasize only one of these roles, this study demonstrates how the same underlying building blocks can produce fundamentally different system behavior depending on architectural intent.

| Architecture   | Strength                | Weakness                       | Best Use Case       |
| -------------- | ----------------------- | ------------------------------ | ------------------- |
| ZDB DLL        | Precise phase alignment | Requires stable feedback       | Clock deskew        |
| Multiphase DLL | Multiple clock phases   | Sensitive to delay mismatch    | SERDES, ADC         |
| TDC-based DLL  | Direct time measurement | Higher complexity, area, power | Calibration systems |
|                |                         |                                |                     |

This comparative approach provides a more complete understanding of the design space and highlights the trade-offs inherent in each implementation. It also reinforces that architectural selection must be driven by system requirements rather than isolated optimization metrics.

---


## Zero-Delay Buffer (ZDB)

**TODO: ADD BLOCK DIAGRAMS + PERFORMANCE INSIGHTS BASED ON GRAPH**

The Zero-Delay Buffer architecture represents the most widely deployed form of DLL in practical digital systems, where the goal is to eliminate the _effective_ delay introduced by clock distribution networks. In a typical ASIC, the clock signal propagates through the clock tree and experiences:

- Buffer delays
- Interconnect (routing) delays
- Load-dependent variations

These effects introduce a deterministic insertion delay (skew) that directly impacts timing margins and synchronization across the system.

---

### Operating Principle

The ZDB addresses this by embedding the delay line within a closed-loop feedback system that compares the input clock with the post-distribution clock. By continuously adjusting the delay line, the system forces alignment between these signals, effectively ensuring:

- The output clock is phase-aligned with the input at the observation point
- The internal delay matches the external clock tree delay

In this sense, the DLL does not remove delay physically, but instead redefines the timing reference so that downstream logic perceives a phase-aligned clock.

---

### Loop Behavior and Non-Idealities

From a control standpoint, the ZDB behaves as a discrete-time feedback system in which phase error is sampled at clock edges and corrected through quantized delay updates. This introduces several important non-idealities:

- **Startup Transient**  
    Large phase errors require multiple correction cycles, making lock time dependent on delay range and step size
    
- **Limit-Cycle Oscillation**  
    In steady state, the system may toggle between adjacent delay codes due to quantization, resulting in bounded jitter
    
- **Stability Trade-off**  
    Larger delay steps improve convergence speed but risk overshoot, while smaller steps improve stability at the cost of longer lock times
    

---

### Architectural Requirements

To ensure reliable operation after synthesis and layout, the architecture must satisfy several constraints:

- Delay updates must be **monotonic**, ensuring smooth convergence without discontinuities
- Phase detection must remain **directionally correct**, even under clock skew and routing mismatch
- The control range must be **bounded**, preventing overflow and undefined states
- Delay switching must be **glitch-free**, avoiding transient pulses that can corrupt the feedback loop
    

---

### Post-Layout Considerations

In practical VLSI implementations, several physical effects influence ZDB behavior:
- Routing symmetry between input and output clocks
- Unequal loading across delay stages
- Clock buffer insertion during synthesis
    

These introduce:

- Residual phase offset even in the locked state
- Variation in effective delay per stage
    
As a result, the design must explicitly account for non-ideal conditions rather than relying on ideal symmetry.

---

Ultimately, the ZDB ensures that all timing decisions are made relative to a compensated clock reference. This makes it indispensable for high-performance synchronous systems, where predictable clock alignment is more critical than minimizing absolute delay.


## Multiphase DLL

**TODO: ADD BLOCK DIAGRAMS + GRAPH BASED INSIGHTS**

The multiphase DLL transforms the delay line from a purely corrective element into a structured phase generation network. Instead of minimizing phase error at a single node, the delay chain is used to construct a discretized representation of time, where each tap corresponds to a specific phase offset relative to the input clock.

---

### Operating Principle

When the total delay of the chain approximates one clock period, each stage produces an evenly spaced phase. This results in a phase lattice that enables the generation of multiple synchronized clock domains from a single input reference. In effect, the delay line becomes a spatial representation of one clock cycle.

This capability is particularly useful in systems such as:

- High-speed serializers/deserializers (SERDES)
- Time-interleaved analog-to-digital converters (ADCs)
- Large-scale clock distribution networks
    

---

### Design Characteristics

Unlike ZDB architectures, where loop stability and convergence dominate design considerations, the multiphase DLL is primarily governed by the uniformity of delay across stages. The accuracy of phase spacing depends on how consistently each delay element behaves under process, voltage, and temperature (PVT) variation.

This introduces several key requirements:

- Delay elements must exhibit **consistent propagation delay** across the chain
- Each tap must drive **similar capacitive loads** to avoid skew distortion
- Physical layout must maintain **routing symmetry** to minimize mismatch
    

---

### Non-Idealities and Error Sources

In practical implementations, several sources of variation degrade phase uniformity:

- Process variation across standard cells
- Unequal capacitive loading between taps
- Interconnect parasitics from routing
    

These effects lead to:

- Non-uniform phase spacing (phase compression or expansion)
- Duty cycle distortion
- Increased jitter in derived clock signals
    

---

### System-Level Impact

Because all phases are used simultaneously, any mismatch in the delay chain produces correlated errors across outputs. These errors directly impact system-level performance:

- In SERDES systems, they reduce sampling margins and timing reliability
- In ADCs, they introduce channel mismatch and degrade signal accuracy

Unlike ZDB architectures, these errors are not corrected through feedback. Instead, they are inherent to the physical realization of the delay structure.

---

The multiphase DLL is best understood not as a control loop, but as a distributed timing fabric. Its performance depends less on convergence behavior and more on how uniformly time is represented across the physical implementation.

As a result, layout symmetry, buffering strategy, and load balancing become primary design factors, often outweighing the importance of the control loop itself.


---

## TDC-Based DLL

**TODO: ADD BLOCK DIAGRAMS** + GRAPH INSIGHTS

The TDC-based DLL represents a fundamental shift in how the delay line is utilized, transforming it from a passive control element into an active time measurement structure. Rather than simply correcting phase error, this architecture enables the system to directly observe and quantify timing differences, effectively converting time into a digital representation.

---

### Operating Principle

In this architecture, the delay chain functions as a time-to-digital converter (TDC). As a signal propagates through the delay line, each stage introduces a fixed delay increment, creating a discrete timeline along the chain. By sampling the outputs of all stages simultaneously, the system captures the position of the signal within this timeline.

This produces a **thermometer code**, where:

- Early stages represent earlier arrival times
- Later stages represent delayed signal propagation
    TODO: ADD THERMOMETER CODE

This code is then converted into a binary or Gray-coded value, yielding a digital estimate of phase difference between signals. In effect, the delay line becomes a spatial encoding of time, allowing the system to measure sub-clock timing differences with fine granularity.

---

### Design Characteristics

The performance of a TDC-based DLL is governed by two tightly coupled factors: resolution and linearity. Resolution is determined by the delay of each stage, with smaller delays enabling finer time measurement. However, achieving high linearity requires uniform delay spacing across all stages, which is difficult to maintain under process, voltage, and temperature (PVT) variation.

This leads to several key design considerations:

- Delay stages must be carefully designed to achieve **consistent propagation delay**
- The delay chain must maintain **monotonic behavior**, ensuring proper thermometer encoding
- Sampling must be synchronized to avoid ambiguity at transition boundaries
- The encoding logic must handle non-ideal inputs without introducing additional errors
    

---

### Non-Idealities and Error Sources

In practical implementations, TDC-based systems are highly sensitive to physical effects that distort timing accuracy. These include:

- **Process variation**, which introduces mismatch between delay stages and causes non-uniform time bins
- **Routing parasitics**, which alter effective delay and distort linearity
- **Supply noise**, which dynamically modulates gate delay and introduces jitter
    

These effects manifest as:

- Non-linear time measurement (unequal bin widths)
- Reduced resolution due to noise and variation
- Increased measurement uncertainty
    

Additionally, TDC architectures are prone to specific digital artifacts:

- **Bubble errors**, where the thermometer code becomes non-monotonic due to delay mismatch
- **Metastability**, particularly when sampling occurs near signal transitions
- **Encoding errors**, if the thermometer-to-binary conversion is not robust
    

---

### System-Level Importance

The importance of TDC-based DLLs extends far beyond simple phase alignment, as they enable a class of systems that rely on _measurement-driven timing control_. In modern VLSI design, this capability is critical for:

- **On-chip calibration systems**, where process variation must be measured and compensated dynamically
- **Adaptive clocking architectures**, which adjust timing based on real-time conditions
- **Jitter measurement and analysis**, allowing systems to monitor and improve clock quality
- **Digital phase-locked loops (DPLLs)**, where TDCs replace analog phase detectors
    

In advanced nodes, where variability and noise dominate, purely open-loop or static designs are insufficient. Instead, systems must continuously measure and adapt, making TDC-based approaches essential for maintaining performance and reliability.

---

### Architectural Implications

Unlike ZDB architectures, where the loop is purely corrective, the TDC-based DLL extracts quantitative timing information that can be used for higher-level decision-making. This fundamentally changes the role of the delay line:

- It becomes a **sensor**, rather than just a control element
- It enables **feedback based on measured data**, not just phase comparison
- It supports **digital calibration loops** that improve system robustness
    

However, this increased functionality comes at the cost of complexity. Additional logic is required for encoding, synchronization, and error correction, all of which increase area and power consumption.

---

### Post-Layout Sensitivity

TDC performance is particularly sensitive to post-layout effects, more so than ZDB or multiphase architectures. Small variations in delay can significantly impact measurement accuracy, making the system highly dependent on:

- Local placement of delay cells
- Matching of interconnect lengths
- Power supply integrity
    

As a result, TDC-based designs often require calibration or compensation mechanisms to maintain accuracy across operating conditions.

---
The TDC-based DLL transforms the delay line into a digital representation of time itself, enabling systems to not only control timing, but also observe and adapt to it.

This makes it fundamentally more powerful than traditional DLL architectures, but also more sensitive to physical implementation details. Its true value lies in enabling measurement-driven design, where timing is no longer assumed to be correct, but is continuously quantified and corrected in real time.


## Core Insight

The key takeaway is that a digital DLL is defined less by ideal timing equations and more by the constraints of real hardware. After synthesis and layout, effects like quantization, routing skew, and cell delay variation dominate behavior, often overriding what is predicted at the model level.

A robust DLL must therefore ensure that all operations are physically realizable: delay updates are monotonic, state transitions are bounded, and phase decisions remain correct under PVT variation. Reliable convergence comes from small, controlled, and implementable corrections—not from ideal control assumptions.

The interaction between blocks is what ultimately determines success. The phase detector must remain directionally correct under skew, the controller must prevent instability or overflow, and the delay line must provide consistent and predictable increments. If any one of these breaks under layout conditions, overall system performance degrades.

These considerations also vary across design flows. In a flow like **Sky130**, variability and limited cell options make robustness and simplicity more critical. In more advanced nodes (e.g., 7nm or below), increased variability, noise, and tighter timing margins make architectures like TDC-based DLLs more attractive due to their ability to measure and adapt. Similarly, analog-heavy flows may rely more on continuous control (PLLs), while fully digital flows prioritize quantized, synthesizable structures.

In the end, a successful DLL is not the one that performs best in theory, but the one that maintains correct and stable behavior across the full physical design flow.